In [1]:
import numpy as np
import pandas as pd

# Updated validated results
results = {
    'Vanilla': {
        'uncompressed': 19.72, 'aac_32': 25.71, 'aac_64': 24.67, 'aac_128': 22.42,
        'opus_16': 75.32, 'opus_32': 82.42, 'opus_64': 71.24
    },
    '+Temporal': {
        'uncompressed': 25.89, 'aac_32': 29.11, 'aac_64': 27.58, 'aac_128': 24.46,
        'opus_16': 78.11, 'opus_32': 46.86, 'opus_64': 46.06
    },
    '+GRL+Con': {
        'uncompressed': 24.66, 'aac_32': 29.83, 'aac_64': 28.56, 'aac_128': 30.61,
        'opus_16': 73.45, 'opus_32': 76.01, 'opus_64': 74.74
    },
    'Full': {
        'uncompressed': 22.95, 'aac_32': 25.79, 'aac_64': 22.94, 'aac_128': 23.23,
        'opus_16': 63.76, 'opus_32': 48.46, 'opus_64': 47.60
    }
}

DAILY_CALLS = 10000
FAKE_RATE   = 0.05
REAL_RATE   = 0.95
daily_fakes = DAILY_CALLS * FAKE_RATE
daily_real  = DAILY_CALLS * REAL_RATE

print(f"Assumptions: {DAILY_CALLS:,} calls/day | {FAKE_RATE*100:.0f}% fake | {REAL_RATE*100:.0f}% real")
print()

# Metric 1: Out of 100 fake calls how many slip through
print("=" * 75)
print("OUT OF 100 FAKE CALLS, HOW MANY SLIP THROUGH?")
print("=" * 75)
print(f"{'Codec':<15}", end="")
for m in results: print(f"{m:>15}", end="")
print()
print("-" * 75)
for codec in results['Vanilla']:
    print(f"{codec:<15}", end="")
    for m in results:
        print(f"{results[m][codec]:>14.1f}%", end="")
    print()

# Metric 2: Daily fraud at scale
print("\n" + "=" * 75)
print(f"FRAUD CALLS SLIPPING THROUGH PER DAY ({DAILY_CALLS:,} calls, {FAKE_RATE*100:.0f}% fake)")
print("=" * 75)
print(f"{'Codec':<15}", end="")
for m in results: print(f"{m:>15}", end="")
print()
print("-" * 75)
for codec in results['Vanilla']:
    print(f"{codec:<15}", end="")
    for m in results:
        fraud = round(daily_fakes * results[m][codec] / 100)
        print(f"{fraud:>15}", end="")
    print()

# Metric 3: Real customers wrongly blocked
print("\n" + "=" * 75)
print(f"REAL CUSTOMERS WRONGLY BLOCKED PER DAY")
print("=" * 75)
print(f"{'Codec':<15}", end="")
for m in results: print(f"{m:>15}", end="")
print()
print("-" * 75)
for codec in results['Vanilla']:
    print(f"{codec:<15}", end="")
    for m in results:
        blocked = round(daily_real * results[m][codec] / 100)
        print(f"{blocked:>15}", end="")
    print()

# Metric 4: Annual fraud prevented by Full vs Vanilla
print("\n" + "=" * 75)
print("ANNUAL FRAUD CALLS PREVENTED: Full vs Vanilla")
print("=" * 75)
total_annual = 0
for codec in results['Vanilla']:
    v = results['Vanilla'][codec]
    f = results['Full'][codec]
    daily_saved  = round(daily_fakes * (v - f) / 100)
    annual_saved = daily_saved * 365
    total_annual += annual_saved
    direction = "prevented" if daily_saved > 0 else "MORE slipping through"
    print(f"  {codec:<15}: {abs(annual_saved):>6,} calls/year {direction}")
print(f"\n  TOTAL: {abs(total_annual):,} fraud calls per year {'prevented' if total_annual > 0 else 'extra'}")

# Metric 5: Deployment verdict
print("\n" + "=" * 75)
print("DEPLOYMENT VERDICT (EER > 33% = unreliable for that codec)")
print("=" * 75)
for m in results:
    deployable   = [c for c, e in results[m].items() if e <= 33]
    undeployable = [c for c, e in results[m].items() if e > 33]
    print(f"  {m}:")
    print(f"    Reliable on   : {', '.join(deployable) if deployable else 'none'}")
    print(f"    Unreliable on : {', '.join(undeployable) if undeployable else 'none'}")

Assumptions: 10,000 calls/day | 5% fake | 95% real

OUT OF 100 FAKE CALLS, HOW MANY SLIP THROUGH?
Codec                  Vanilla      +Temporal       +GRL+Con           Full
---------------------------------------------------------------------------
uncompressed             19.7%          25.9%          24.7%          22.9%
aac_32                   25.7%          29.1%          29.8%          25.8%
aac_64                   24.7%          27.6%          28.6%          22.9%
aac_128                  22.4%          24.5%          30.6%          23.2%
opus_16                  75.3%          78.1%          73.5%          63.8%
opus_32                  82.4%          46.9%          76.0%          48.5%
opus_64                  71.2%          46.1%          74.7%          47.6%

FRAUD CALLS SLIPPING THROUGH PER DAY (10,000 calls, 5% fake)
Codec                  Vanilla      +Temporal       +GRL+Con           Full
---------------------------------------------------------------------------
unco

In [2]:
# Metric 6: Breaking point
print("=" * 75)
print("AT WHAT COMPRESSION DOES EACH MODEL BREAK?")
print("(breaking = worse than 1 in 2 calls wrong, EER > 50%)")
print("=" * 75)
for m in results:
    broken = [c for c, e in results[m].items() if e > 50]
    ok     = [c for c, e in results[m].items() if e <= 50]
    print(f"  {m}:")
    print(f"    Survives : {', '.join(ok) if ok else 'none'}")
    print(f"    Breaks   : {', '.join(broken) if broken else 'none'}")

# Metric 7: Consistency
print("\n" + "=" * 75)
print("MODEL CONSISTENCY ACROSS CODECS")
print("(lower std = more predictable in production)")
print("=" * 75)
for m in results:
    eers = list(results[m].values())
    print(f"  {m}:")
    print(f"    Best case  : {min(eers):.1f}% ({min(results[m], key=results[m].get)})")
    print(f"    Worst case : {max(eers):.1f}% ({max(results[m], key=results[m].get)})")
    print(f"    Std dev    : {np.std(eers):.1f}% — {'unpredictable' if np.std(eers) > 20 else 'consistent'}")

# Metric 8: Recovery rate Full vs Vanilla
print("\n" + "=" * 75)
print("HOW MUCH DID FULL MODEL RECOVER VS VANILLA?")
print("=" * 75)
for codec in results['Vanilla']:
    v        = results['Vanilla'][codec]
    f        = results['Full'][codec]
    recovery = v - f
    pct      = (recovery / v) * 100
    direction = f"{abs(pct):.1f}% {'improvement' if recovery > 0 else 'regression'}"
    print(f"  {codec:<15}: {recovery:+.2f}% EER ({direction})")

# Metric 9: Best model per codec
print("\n" + "=" * 75)
print("WHICH MODEL IS BEST FOR EACH CODEC?")
print("=" * 75)
for codec in results['Vanilla']:
    best_model = min(results, key=lambda m: results[m][codec])
    best_eer   = results[best_model][codec]
    worst_model = max(results, key=lambda m: results[m][codec])
    worst_eer   = results[worst_model][codec]
    print(f"  {codec:<15}: Best={best_model} ({best_eer:.1f}%)  Worst={worst_model} ({worst_eer:.1f}%)")

# Metric 10: Overall ranking
print("\n" + "=" * 75)
print("OVERALL MODEL RANKING (by average EER across all codecs)")
print("=" * 75)
ranking = sorted(results.items(), key=lambda x: np.mean(list(x[1].values())))
for rank, (m, data) in enumerate(ranking, 1):
    avg = np.mean(list(data.values()))
    print(f"  #{rank} {m}: avg EER = {avg:.1f}%")

# Metric 11: Weighted score (Opus matters more in production)
print("\n" + "=" * 75)
print("PRODUCTION-WEIGHTED SCORE (Opus weighted 3x, others 1x)")
print("(Opus is the dominant VoIP codec in production)")
print("=" * 75)
weights = {
    'uncompressed': 1, 'aac_32': 1, 'aac_64': 1, 'aac_128': 1,
    'opus_16': 3, 'opus_32': 3, 'opus_64': 3
}
for m in results:
    weighted = sum(results[m][c] * weights[c] for c in results[m])
    total_w  = sum(weights.values())
    wavg     = weighted / total_w
    print(f"  {m}: weighted EER = {wavg:.1f}%")

AT WHAT COMPRESSION DOES EACH MODEL BREAK?
(breaking = worse than 1 in 2 calls wrong, EER > 50%)
  Vanilla:
    Survives : uncompressed, aac_32, aac_64, aac_128
    Breaks   : opus_16, opus_32, opus_64
  +Temporal:
    Survives : uncompressed, aac_32, aac_64, aac_128, opus_32, opus_64
    Breaks   : opus_16
  +GRL+Con:
    Survives : uncompressed, aac_32, aac_64, aac_128
    Breaks   : opus_16, opus_32, opus_64
  Full:
    Survives : uncompressed, aac_32, aac_64, aac_128, opus_32, opus_64
    Breaks   : opus_16

MODEL CONSISTENCY ACROSS CODECS
(lower std = more predictable in production)
  Vanilla:
    Best case  : 19.7% (uncompressed)
    Worst case : 82.4% (opus_32)
    Std dev    : 26.6% — unpredictable
  +Temporal:
    Best case  : 24.5% (aac_128)
    Worst case : 78.1% (opus_16)
    Std dev    : 17.9% — consistent
  +GRL+Con:
    Best case  : 24.7% (uncompressed)
    Worst case : 76.0% (opus_32)
    Std dev    : 23.0% — unpredictable
  Full:
    Best case  : 22.9% (aac_64)
    Wor